# Assignment 08: CLIP Contrastive Loss (100 points)

**Unit 10: Computer Vision & Generative AI | AI 520**

**CRITICAL**: This assignment mirrors the 2025 USAAIO Round 2 Problem 3 structure.
Implement the full CLIP training pipeline: dual encoders, projection heads, cosine similarity, InfoNCE loss, and training loop.

**Format**: USAAIO Round 2 Style

In [ ]:
# DO NOT CHANGE THIS CELL
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

---
**WARNING**: Do not import additional libraries. Implement everything from scratch using only `torch`.

---

## Part 1: Image Encoder (8 points)

Implement a simple image encoder (simplified, not a full ViT).

Architecture: `Flatten -> Linear(img_dim, 512) -> ReLU -> Linear(512, 256) -> ReLU -> Linear(256, feature_dim)`

**Input**: `(B, C, H, W)` images
**Output**: `(B, feature_dim)` image features (NOT yet projected or normalized)

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, img_dim: int = 784, feature_dim: int = 128):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # images: (B, C, H, W) -> flatten -> (B, feature_dim)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 2: Text Encoder (8 points)

Implement a simple text encoder.

Architecture: `Linear(text_dim, 256) -> ReLU -> Linear(256, feature_dim)`

**Input**: `(B, text_dim)` text features (pre-computed, e.g., bag of words)
**Output**: `(B, feature_dim)` text features

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, text_dim: int = 300, feature_dim: int = 128):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, text_features: torch.Tensor) -> torch.Tensor:
        # text_features: (B, text_dim) -> (B, feature_dim)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 3: Projection Head (8 points)

Project encoder features to a shared embedding space.

Architecture: `Linear(feature_dim, embed_dim)` (no bias)

**Output**: L2-normalized embeddings of shape `(B, embed_dim)`

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, feature_dim: int = 128, embed_dim: int = 64):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, features: torch.Tensor) -> torch.Tensor:
        # features: (B, feature_dim) -> project -> L2 normalize -> (B, embed_dim)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 4: Cosine Similarity Matrix (10 points)

Compute the cosine similarity matrix between image and text embeddings.

Since embeddings are already L2-normalized, cosine similarity = dot product.

**Input**: `image_embeds` and `text_embeds`, each `(N, D)`, L2-normalized
**Output**: `(N, N)` similarity matrix

In [ ]:
def cosine_similarity_matrix(image_embeds: torch.Tensor, text_embeds: torch.Tensor) -> torch.Tensor:
    """
    Compute cosine similarity matrix.
    image_embeds: (N, D) L2-normalized
    text_embeds: (N, D) L2-normalized
    Returns: (N, N) where entry (i,j) = cos(image_i, text_j)
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 5: Temperature-Scaled Logits (8 points)

Scale the similarity matrix by a learnable temperature parameter $\tau$.

$$\text{logits}_{ij} = \frac{\text{sim}(v_i, t_j)}{\tau}$$

Temperature is stored as $\log(1/\tau)$ for numerical stability.

In [ ]:
def compute_logits(similarity: torch.Tensor, log_inv_temperature: torch.Tensor) -> torch.Tensor:
    """
    Scale similarity matrix by temperature.
    similarity: (N, N) cosine similarities
    log_inv_temperature: scalar parameter = log(1/tau)
    Returns: (N, N) scaled logits
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 6: InfoNCE Loss — Image to Text (10 points)

For each image $i$, compute the cross-entropy loss treating the matching text $i$ as the correct class:

$$\mathcal{L}_i^{i2t} = -\log \frac{\exp(\text{logits}_{ii})}{\sum_{k=1}^{N} \exp(\text{logits}_{ik})}$$

Average over all images in the batch.

In [ ]:
def infonce_image_to_text(logits: torch.Tensor) -> torch.Tensor:
    """
    Image-to-text InfoNCE loss.
    logits: (N, N) scaled similarity matrix
    Labels are the diagonal: image i matches text i.
    Returns: scalar loss
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 7: InfoNCE Loss — Text to Image (10 points)

Symmetric: for each text $j$, the matching image $j$ is the correct class.

$$\mathcal{L}_j^{t2i} = -\log \frac{\exp(\text{logits}_{jj})}{\sum_{k=1}^{N} \exp(\text{logits}_{kj})}$$

In [ ]:
def infonce_text_to_image(logits: torch.Tensor) -> torch.Tensor:
    """
    Text-to-image InfoNCE loss.
    logits: (N, N) scaled similarity matrix
    Returns: scalar loss
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 8: Symmetric CLIP Loss (8 points)

$$\mathcal{L}_{\text{CLIP}} = \frac{1}{2}(\mathcal{L}^{i2t} + \mathcal{L}^{t2i})$$

In [ ]:
def clip_loss(logits: torch.Tensor) -> torch.Tensor:
    """
    Symmetric CLIP loss.
    logits: (N, N) scaled similarity matrix
    Returns: scalar loss
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 9: Full CLIP Model (10 points)

Assemble the full CLIP model with image encoder, text encoder, projection heads, and learnable temperature.

In [ ]:
class CLIPModel(nn.Module):
    def __init__(
        self,
        img_dim: int = 784,
        text_dim: int = 300,
        feature_dim: int = 128,
        embed_dim: int = 64,
        temperature_init: float = 0.07
    ):
        super().__init__()
        # Create: image_encoder, text_encoder, image_proj, text_proj, log_inv_temperature
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, images: torch.Tensor, texts: torch.Tensor) -> torch.Tensor:
        """
        images: (N, C, H, W)  texts: (N, text_dim)
        Returns: (N, N) logits matrix (scaled by temperature)
        """
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 10: CLIP Training Step (10 points)

Implement a single training step for CLIP.

In [ ]:
def clip_train_step(
    model: CLIPModel,
    images: torch.Tensor,
    texts: torch.Tensor,
    optimizer: torch.optim.Optimizer
) -> dict:
    """
    Single CLIP training step.
    Returns: dict with 'loss', 'temperature' (current tau value)
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 11: Retrieval Accuracy (10 points)

Compute image-to-text retrieval accuracy: for each image, check if the highest-similarity text is the correct match.

$$\text{Accuracy} = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[\arg\max_k S_{ik} = i]$$

In [ ]:
@torch.no_grad()
def retrieval_accuracy(model: CLIPModel, images: torch.Tensor, texts: torch.Tensor) -> dict:
    """
    Compute image-to-text and text-to-image retrieval accuracy.
    Returns: dict with 'i2t_accuracy', 't2i_accuracy'
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 12: Numeric Verification (8 points)

Given specific embeddings, compute the CLIP loss by hand and verify your implementation matches.

Use the embeddings below to verify Parts 4-8.

In [ ]:
# DO NOT CHANGE THIS CELL — use it to verify your implementation
# Batch of 3 image-text pairs
image_embeds = F.normalize(torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
]), dim=-1)  # already orthonormal

text_embeds = F.normalize(torch.tensor([
    [0.9, 0.1, 0.0],
    [0.1, 0.9, 0.1],
    [0.0, 0.1, 0.9]
]), dim=-1)

tau = 0.07
log_inv_tau = torch.tensor(np.log(1.0 / tau))

# Your code should produce:
sim = cosine_similarity_matrix(image_embeds, text_embeds)
logits = compute_logits(sim, log_inv_tau)
loss = clip_loss(logits)
print(f"Similarity matrix:\n{sim}")
print(f"Logits:\n{logits}")
print(f"CLIP loss: {loss.item():.4f}")